# Murmur GQA 40M — English smoke

Control architecture smoke. It uses the same automatic FineWeb-Edu English profile as SISO and MIMO; only the mixer/config differs.

In [ ]:
from pathlib import Path
import subprocess, sys, torch
REPO=Path('/workspace/murmur-science')
if not REPO.exists(): subprocess.run(['git','clone','--branch','codex/dataset-mix-notebooks','https://github.com/orkrs/murmur-science.git',str(REPO)],check=True)
%cd /workspace/murmur-science
sys.path.insert(0,str(Path.cwd()/'src'))
if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU is required')
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
%pip install -q datasets sentencepiece pyarrow pandas einops ninja
!python scripts/param_count.py --config configs/smoke_gqa.toml
from murmur.config import load_run_config
config=load_run_config(Path('configs/smoke_gqa.toml'))
assert config.model.mixer=='gqa'
print('Verified GQA control configuration')

In [ ]:
# English-only FineWeb-Edu smoke corpus
subprocess.run([sys.executable,'scripts/build_hf_mix.py','--profile','english_smoke','--output','artifacts/english_smoke_corpus','--max-tokens','2400000'],check=True)
assert Path('artifacts/english_smoke_corpus/train.jsonl').exists()
print(Path('artifacts/english_smoke_corpus/provenance.json').read_text(encoding='utf-8'))

In [ ]:
!python scripts/train_tokenizer.py --corpus artifacts/english_smoke_corpus/corpus.txt --output artifacts/english_smoke_tokenizer.model --vocab-size 32000
!python scripts/prepare_data.py --config configs/smoke_gqa.toml --tokenizer artifacts/english_smoke_tokenizer.model --train-input artifacts/english_smoke_corpus/train.jsonl --val-input artifacts/english_smoke_corpus/val.jsonl --output artifacts/english_smoke_data
template=Path('configs/smoke_gqa.toml').read_text()
Path('configs/smoke_gqa_session.toml').write_text(template.replace('artifacts/data/train.bin','artifacts/english_smoke_data/train.bin').replace('artifacts/data/val.bin','artifacts/english_smoke_data/val.bin'))

In [ ]:
run_dir=Path('artifacts/runs/gqa_english_smoke')
subprocess.run([sys.executable,'scripts/train.py','--config','configs/smoke_gqa_session.toml','--run-dir',str(run_dir),'--device','cuda'],check=True)
assert (run_dir/'checkpoints'/'last'/'COMPLETED').exists()
print('GQA English smoke checkpoint ready')